In [1]:
"""
==================================================================================
 CUSTOMER RETENTION INTELLIGENCE PLATFORM - SECONDARY DATASET (Olist)
 Phase 2: Data Cleaning -> olist_customers_dataset.csv (for Tableau dashboard)
==================================================================================
 Purpose : Clean the customer dimension table for use in Tableau, joined to
           olist_orders_dataset.csv via 'customer_id'.

 IMPORTANT NOTES FOR A BI/DASHBOARD DATASET:
   1. 'customer_id' is unique per ORDER and is the join key to the orders
      table -> it must stay untouched and fully unique. Confirmed clean below.
   2. 'customer_unique_id' identifies the actual PERSON -> it is expected to
      repeat for customers who ordered more than once. These repeats are NOT
      duplicates to be removed; removing them would silently delete real
      orders from the dashboard. They are kept as-is and simply documented.
   3. This file had no missing values or duplicate rows -> cleaning here is
      mostly standardization (casing/formatting) for consistent Tableau
      labels, plus validation, rather than heavy repair.
==================================================================================
"""

import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 150)


def section(title: str) -> None:
    print("\n" + "=" * 90)
    print(f" {title}")
    print("=" * 90)


# ----------------------------------------------------------------------------------
# 1. LOAD RAW DATA
# ----------------------------------------------------------------------------------
section("1. LOAD RAW DATA")

RAW_PATH = "/Users/meetnakrani/Library/Mobile Documents/com~apple~CloudDocs/Customer-Retention-intelligence-Platform/DataSet/RAW/secondary Data/olist_customers_dataset.csv"
CLEANED_PATH = "/Users/meetnakrani/Library/Mobile Documents/com~apple~CloudDocs/Customer-Retention-intelligence-Platform/DataSet/cleaned/olist_customers_cleaned.csv"

df = pd.read_csv(RAW_PATH)
print(f"Raw dataset loaded: {df.shape[0]} rows, {df.shape[1]} columns")


# ----------------------------------------------------------------------------------
# 2. DUPLICATE RECORD CHECK (informational only - see notes above)
# ----------------------------------------------------------------------------------
section("2. DUPLICATE RECORD CHECK")

full_dupes = df.duplicated().sum()
id_dupes = df["customer_id"].duplicated().sum()
person_repeats = df["customer_unique_id"].duplicated().sum()

print(f"Fully duplicated rows                  : {full_dupes}")
print(f"Duplicated customer_id (join key)      : {id_dupes}  <- must be 0, confirmed below")
print(f"Repeated customer_unique_id (person)    : {person_repeats}  <- expected, NOT removed")

before_rows = df.shape[0]
df = df.drop_duplicates()   # safety net; confirmed 0 found, no rows actually removed
after_rows = df.shape[0]
print(f"\nRows removed as full duplicates        : {before_rows - after_rows}")


# ----------------------------------------------------------------------------------
# 3. MISSING VALUE CHECK
# ----------------------------------------------------------------------------------
section("3. MISSING VALUE CHECK")

missing = df.isnull().sum()
missing = missing[missing > 0]
if missing.empty:
    print("No missing values found. No imputation needed for this file.")
else:
    print(missing)


# ----------------------------------------------------------------------------------
# 4. TEXT STANDARDIZATION (city / state formatting for clean Tableau labels)
# ----------------------------------------------------------------------------------
section("4. TEXT STANDARDIZATION")

print("Before standardization (sample):")
print(df["customer_city"].head(5).tolist())

# City names arrive in lowercase Portuguese (e.g. "sao paulo") -> title-case
# them for readable Tableau labels/tooltips. State codes are already clean
# 2-letter uppercase codes, so they are left untouched.
df["customer_city"] = df["customer_city"].str.strip().str.title()
df["customer_state"] = df["customer_state"].str.strip().str.upper()

print("\nAfter standardization (sample):")
print(df["customer_city"].head(5).tolist())


# ----------------------------------------------------------------------------------
# 5. DATA TYPE & KEY VALIDATION
# ----------------------------------------------------------------------------------
section("5. DATA TYPE & KEY VALIDATION")

print(df.dtypes)
print(f"\ncustomer_zip_code_prefix range: {df['customer_zip_code_prefix'].min()} "
      f"to {df['customer_zip_code_prefix'].max()}")
print(f"Unique states: {df['customer_state'].nunique()} "
      f"(Brazil has 27 states + Federal District -> should match)")


# ----------------------------------------------------------------------------------
# 6. FINAL VALIDATION
# ----------------------------------------------------------------------------------
section("6. FINAL VALIDATION")

print(f"Final shape                      : {df.shape[0]} rows, {df.shape[1]} columns")
print(f"Remaining missing values         : {df.isnull().sum().sum()}")
print(f"Remaining full duplicate rows    : {df.duplicated().sum()}")
print(f"customer_id still fully unique   : {df['customer_id'].is_unique}")
print(f"Row count unchanged from raw file: {df.shape[0] == before_rows}")


# ----------------------------------------------------------------------------------
# 7. EXPORT CLEANED DATASET
# ----------------------------------------------------------------------------------
section("7. EXPORT CLEANED DATASET")

df.to_csv(CLEANED_PATH, index=False)
print(f"Cleaned dataset saved to: {CLEANED_PATH}")

section("CUSTOMERS DATASET CLEANING COMPLETE")
print("This file is now safe to load into Tableau and join to orders on 'customer_id'.")
print("Remember: use 'customer_unique_id' (not 'customer_id') for any true")
print("customer-count or repeat-customer KPI, since customer_id repeats per order.")


 1. LOAD RAW DATA
Raw dataset loaded: 99441 rows, 5 columns

 2. DUPLICATE RECORD CHECK
Fully duplicated rows                  : 0
Duplicated customer_id (join key)      : 0  <- must be 0, confirmed below
Repeated customer_unique_id (person)    : 3345  <- expected, NOT removed

Rows removed as full duplicates        : 0

 3. MISSING VALUE CHECK
No missing values found. No imputation needed for this file.

 4. TEXT STANDARDIZATION
Before standardization (sample):
['franca', 'sao bernardo do campo', 'sao paulo', 'mogi das cruzes', 'campinas']

After standardization (sample):
['Franca', 'Sao Bernardo Do Campo', 'Sao Paulo', 'Mogi Das Cruzes', 'Campinas']

 5. DATA TYPE & KEY VALIDATION
customer_id                   str
customer_unique_id            str
customer_zip_code_prefix    int64
customer_city                 str
customer_state                str
dtype: object

customer_zip_code_prefix range: 1003 to 99990
Unique states: 27 (Brazil has 27 states + Federal District -> should match)
